In [1]:
import glob
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships


In [2]:
# The 294 genomes and, for each, the reads it was actually assembled from. The roster
# carries the exact read files the assembly ran on - for four strains the ONT set is the
# Kraken2-decontaminated one from 06_asm_fixing, and quoting yield off the raw library
# would describe reads that never reached Flye.
roster = pd.read_csv(TABLES / "KASPAH_QC_outcome_v2.tsv", sep="\t")
roster = roster[roster["assemble"] == True].set_index("sample")
final_qc = pd.read_csv(TABLES / "final_qc_summary.tsv", sep="\t")

assert set(final_qc["library"]) == set(roster.index)
assert (final_qc["final_qc_decision"] == "PASS").all()
len(final_qc), roster["read_prep"].value_counts().to_dict()

(294, {'standard': 290, 'kraken2_decontam': 4})

In [3]:
# ONT yield. ../pipeline_tables/ont_reads.stats.tsv is seqkit over the v1 libraries and names files
# relative to 02_reads/; ont_read_stats.sh measures what it does not cover - the 19 _v2
# libraries and the 4 decontaminated read sets - and names them project-relative.
ont = pd.concat([
    pd.read_csv(TABLES / "ont_reads.stats.tsv", sep="\t").assign(
        file=lambda df: "02_reads/" + df["file"]),
    pd.read_csv(TABLES / "ont_reads.stats_v2.tsv", sep="\t"),
])
ont = ont.set_index("file")[["num_seqs", "sum_len", "avg_len", "N50", "AvgQual", "Q20(%)", "GC(%)"]]

unmeasured = sorted(set(roster["ont_reads"]) - set(ont.index))
assert not unmeasured, f"run 11_manuscript/ont_read_stats.sh: {unmeasured}"

ont_yield = (ont.loc[roster["ont_reads"]]
             .set_axis(roster.index)
             .rename(columns={"num_seqs": "ont_reads", "sum_len": "ont_bases",
                              "avg_len": "ont_mean_read_len", "N50": "ont_read_n50",
                              "AvgQual": "ont_mean_qual", "Q20(%)": "ont_q20_pct",
                              "GC(%)": "ont_gc_pct"}))
ont_yield.head(3)

,ont_reads,ont_bases,ont_mean_read_len,ont_read_n50,ont_mean_qual,ont_q20_pct,ont_gc_pct
sample,,,,,,,
INF001.1,30641,446886340,14584.6,21312,16.11,84,57.21
INF002,87144,659309449,7565.7,11207,13.53,75,56.76
INF003,56726,874671079,15419.2,23077,18.14,89,56.99


In [4]:
# Illumina yield, from the fastp report written beside the reads. fastp counts R1 and R2
# together, so its total_reads is the pair count doubled. v1 named the report
# <sample>_fastp.json and v2 fastp_report.json, so the file is globbed rather than named.
def fastp_yield(illumina_r1):
    report, = sorted(Path("..", illumina_r1).parent.glob("*fastp*.json"))
    summary = json.loads(report.read_text())["summary"]
    return pd.Series({
        "illumina_raw_read_pairs": summary["before_filtering"]["total_reads"] // 2,
        "illumina_read_pairs": summary["after_filtering"]["total_reads"] // 2,
        "illumina_bases": summary["after_filtering"]["total_bases"],
        "illumina_read_len": summary["after_filtering"]["read1_mean_length"],
        "illumina_q30_pct": 100 * summary["after_filtering"]["q30_rate"],
        "illumina_gc_pct": 100 * summary["after_filtering"]["gc_content"],
        "illumina_pct_reads_kept": (100 * summary["after_filtering"]["total_reads"]
                                    / summary["before_filtering"]["total_reads"]),
    })


illumina_yield = roster["illumina_r1"].apply(fastp_yield)
illumina_yield.head(3)

,illumina_raw_read_pairs,illumina_read_pairs,illumina_bases,illumina_read_len,illumina_q30_pct,illumina_gc_pct,illumina_pct_reads_kept
sample,,,,,,,
INF001.1,3342112.0,3253710.0,813335422.0,124.0,94.1649,56.6603,97.354906
INF002,3656438.0,3560758.0,890011062.0,124.0,94.2855,56.5762,97.383246
INF003,2409084.0,2337265.0,584291252.0,124.0,93.6965,56.1738,97.018825


In [5]:
# Contig lengths and topology read straight out of the genomes rather than off a summary
# table. Flye and clinopore-nf write circular=Y|N into the header; the one Autocycler
# assembly (INF205) carries no such flag, and its contigs come back as None so that they
# are reported as unknown instead of being counted either way.
def contigs_of(isolate):
    lengths, circular, gc, length, bases_gc = [], [], [], 0, 0
    for line in open(f"../10_final_assemblies/assemblies/{isolate}.fasta"):
        if line.startswith(">"):
            if length:
                lengths.append(length)
                gc.append(bases_gc)
            flag = re.search(r"circular=([YN])", line)
            circular.append({"Y": True, "N": False}[flag.group(1)] if flag else None)
            length, bases_gc = 0, 0
        else:
            sequence = line.strip().upper()
            length += len(sequence)
            bases_gc += sequence.count("G") + sequence.count("C")
    lengths.append(length)
    gc.append(bases_gc)
    return (pd.DataFrame({"length": lengths, "circular": circular, "gc_bases": gc})
            .sort_values("length", ascending=False, ignore_index=True))


contigs_of("INF002")

,length,circular,gc_bases
0,5425360,True,3114280
1,183856,True,96617
2,18080,True,9943


In [6]:
# Per-assembly structure. The chromosome is the longest contig, which 09_final_qc already
# established is the only one above 4 Mb in every genome; everything else is plasmid or
# other extrachromosomal sequence.
CHROMOSOME_MIN_LEN = 4_000_000          # the 09_final_qc definition


def assembly_stats(isolate):
    contigs = contigs_of(isolate)
    chromosome = contigs.iloc[0]
    assert chromosome["length"] >= CHROMOSOME_MIN_LEN, isolate
    assert (contigs["length"][1:] < CHROMOSOME_MIN_LEN).all(), isolate
    closed = contigs["circular"] == True
    return pd.Series({
        "contigs": len(contigs),
        "total_len": contigs["length"].sum(),
        "mean_contig_len": contigs["length"].mean(),
        "chromosome_len": chromosome["length"],
        "chromosome_circular": {True: "Y", False: "N", None: "unknown"}[chromosome["circular"]],
        "chromosome_gc_pct": 100 * chromosome["gc_bases"] / chromosome["length"],
        "assembly_gc_pct": 100 * contigs["gc_bases"].sum() / contigs["length"].sum(),
        "plasmid_contigs": len(contigs) - 1,
        "circular_contigs": int(closed.sum()),
        "single_contig": len(contigs) == 1,
        "fully_circular": bool(closed.all()),
    })


structure = final_qc.set_index("library")["isolate"].apply(assembly_stats)
structure.head(3)

,contigs,total_len,mean_contig_len,chromosome_len,chromosome_circular,chromosome_gc_pct,assembly_gc_pct,plasmid_contigs,circular_contigs,single_contig,fully_circular
library,,,,,,,,,,,
INF001.1,2,5658607,2.829304e+06,5571684,Y,57.407240,57.337027,1,2,False,True
INF002,3,5627296,1.875765e+06,5425360,Y,57.402274,57.236015,2,3,False,True
INF003,3,5534757,1.844919e+06,5232791,Y,57.584471,57.233461,2,3,False,True


In [7]:
# One row per genome: what was sequenced, what came out, and what it types as. Depth is
# yield over assembly length, which for these genomes is within a per cent of genome size.
typing = (final_qc.set_index("library")[
              ["isolate", "resequenced", "hybrid_species", "hybrid_ST", "hybrid_K_locus",
               "hybrid_K_locus_confidence", "hybrid_O_locus"]]
          .rename(columns=lambda column: column.replace("hybrid_", "")))

genomes = typing.join(ont_yield).join(illumina_yield).join(structure)
genomes["ont_depth"] = genomes["ont_bases"] / genomes["total_len"]
genomes["illumina_depth"] = genomes["illumina_bases"] / genomes["total_len"]

assert len(genomes) == genomes["isolate"].nunique() == 294
assert genomes.notna().all().all()
genomes.head(3)

,isolate,resequenced,species,ST,K_locus,K_locus_confidence,O_locus,ont_reads,ont_bases,ont_mean_read_len,ont_read_n50,ont_mean_qual,ont_q20_pct,ont_gc_pct,illumina_raw_read_pairs,illumina_read_pairs,illumina_bases,illumina_read_len,illumina_q30_pct,illumina_gc_pct,illumina_pct_reads_kept,contigs,total_len,mean_contig_len,chromosome_len,chromosome_circular,chromosome_gc_pct,assembly_gc_pct,plasmid_contigs,circular_contigs,single_contig,fully_circular,ont_depth,illumina_depth
library,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
INF001.1,INF001,False,Klebsiella variicola subsp. variicola,ST3090,KL3,Typeable,OL3α/β,30641,446886340,14584.6,21312,16.11,84,57.21,3342112.0,3253710.0,813335422.0,124.0,94.1649,56.6603,97.354906,2,5658607,2.829304e+06,5571684,Y,57.407240,57.337027,1,2,False,True,78.974620,143.734213
INF002,INF002,False,Klebsiella pneumoniae,ST133,KL116,Typeable,OL2α.1,87144,659309449,7565.7,11207,13.53,75,56.76,3656438.0,3560758.0,890011062.0,124.0,94.2855,56.5762,97.383246,3,5627296,1.875765e+06,5425360,Y,57.402274,57.236015,2,3,False,True,117.162745,158.159632
INF003,INF003,False,Klebsiella pneumoniae,ST323,KL21,Typeable,OL3γ,56726,874671079,15419.2,23077,18.14,89,56.99,2409084.0,2337265.0,584291252.0,124.0,93.6965,56.1738,97.018825,3,5534757,1.844919e+06,5232791,Y,57.584471,57.233461,2,3,False,True,158.032427,105.567643


In [8]:
# Sequencing depth and assembly contiguity across the cohort. ONT read counts vary an
# order of magnitude between libraries because barcoded runs do not split evenly, so the
# median is the honest headline and the minimum is worth quoting beside it.
SPREAD = ["ont_reads", "ont_bases", "ont_mean_read_len", "ont_read_n50", "ont_mean_qual",
          "ont_q20_pct", "ont_depth",
          "illumina_read_pairs", "illumina_bases", "illumina_read_len", "illumina_q30_pct",
          "illumina_pct_reads_kept", "illumina_depth",
          "contigs", "mean_contig_len", "total_len", "chromosome_len", "chromosome_gc_pct",
          "plasmid_contigs"]

spread = genomes[SPREAD].describe().T[["mean", "50%", "min", "max", "std"]]
spread = spread.rename(columns={"50%": "median"}).round(1)
spread

,mean,median,min,max,std
ont_reads,76576.9,66579.5,4734.0,5.671290e+05,54620.5
ont_bases,914341838.7,841306147.0,76639141.0,3.922231e+09,517733755.9
ont_mean_read_len,12955.7,13134.4,3873.0,2.225230e+04,2852.6
ont_read_n50,18214.9,18325.0,4467.0,3.623000e+04,4762.7
ont_mean_qual,16.8,17.0,9.6,1.980000e+01,1.4
ont_q20_pct,85.9,87.0,42.0,9.200000e+01,5.4
ont_depth,165.3,151.0,14.1,6.610000e+02,92.1
illumina_read_pairs,2742364.0,2578010.0,1167252.0,7.795268e+06,816889.2
illumina_bases,702425993.9,649082381.0,290303978.0,2.136559e+09,232421353.1
illumina_read_len,127.3,124.0,124.0,2.810000e+02,17.8


In [9]:
# How finished the genomes are. "Complete" in the single-contig sense is the strict
# reading - chromosome and every plasmid in one closed piece - and is much rarer than a
# closed chromosome, because most of these isolates carry plasmids and a few carry many.
closed = pd.Series({
    "isolates": len(genomes),
    "circular chromosome": (genomes["chromosome_circular"] == "Y").sum(),
    "chromosome topology unknown": (genomes["chromosome_circular"] == "unknown").sum(),
    "single-contig assembly": genomes["single_contig"].sum(),
    "every contig circular": genomes["fully_circular"].sum(),
    ">= 1 extrachromosomal contig": (genomes["plasmid_contigs"] > 0).sum(),
})
completeness = pd.DataFrame({"n": closed, "pct": (100 * closed / len(genomes)).round(1)})
completeness

,n,pct
isolates,294,100.0
circular chromosome,276,93.9
chromosome topology unknown,1,0.3
single-contig assembly,60,20.4
every contig circular,224,76.2
>= 1 extrachromosomal contig,234,79.6


In [10]:
# The lineage diversity behind the manuscript table. Most STs are seen once, which is what
# a collection assembled from unrelated clinical isolates should look like.
diversity = pd.concat([
    genomes[column].value_counts().rename_axis("value").reset_index(name="n_isolates").assign(type=label)
    for column, label in [("ST", "ST"), ("K_locus", "K_locus"), ("O_locus", "O_locus")]
], ignore_index=True)
diversity["pct_isolates"] = (100 * diversity["n_isolates"] / len(genomes)).round(2)
diversity = diversity[["type", "value", "n_isolates", "pct_isolates"]]

pd.DataFrame({
    "distinct": diversity.groupby("type")["value"].count(),
    "singletons": diversity[diversity["n_isolates"] == 1].groupby("type")["value"].count(),
    "most common": diversity.loc[diversity.groupby("type")["n_isolates"].idxmax()].set_index("type")["value"],
    "n": diversity.groupby("type")["n_isolates"].max(),
})

,distinct,singletons,most common,n
type,,,,
K_locus,96,40,KL30,15
O_locus,10,1,OL2α.1,80
ST,212,177,ST323,8


In [11]:
# THE MANUSCRIPT TABLE: rows are species, columns are cohort size, lineage diversity and
# how many assemblies came out complete. O loci are carried in per_isolate.tsv and
# diversity_counts.tsv but left out here - the manuscript reports K types and STs.
species = genomes.groupby("species").agg(
    n_isolates=("isolate", "size"),
    n_distinct_ST=("ST", "nunique"),
    n_distinct_K_locus=("K_locus", "nunique"),
    n_circular_chromosome=("chromosome_circular", lambda flag: (flag == "Y").sum()),
    n_fully_circular=("fully_circular", "sum"),
    n_single_contig=("single_contig", "sum"),
    median_contigs=("contigs", "median"),
    median_total_len=("total_len", "median"),
    median_chromosome_len=("chromosome_len", "median"),
).sort_values("n_isolates", ascending=False)

species.loc["All"] = species.sum()
for column in ["n_distinct_ST", "n_distinct_K_locus"]:
    species.loc["All", column] = genomes[column.replace("n_distinct_", "")].nunique()
for column in ["median_contigs", "median_total_len", "median_chromosome_len"]:
    species.loc["All", column] = genomes[column.replace("median_", "")].median()

species = species.astype(int, errors="ignore")
species

,n_isolates,n_distinct_ST,n_distinct_K_locus,n_circular_chromosome,n_fully_circular,n_single_contig,median_contigs,median_total_len,median_chromosome_len
species,,,,,,,,,
Klebsiella pneumoniae,224,155,80,211,172,45,2,5499944,5282955
Klebsiella variicola subsp. variicola,56,46,31,53,42,12,2,5700309,5496302
Klebsiella quasipneumoniae subsp. similipneumoniae,11,8,8,9,7,1,3,5432257,5141367
Klebsiella quasipneumoniae subsp. quasipneumoniae,3,3,2,3,3,2,1,5295856,5295856
All,294,212,96,276,224,60,2,5519564,5307218


In [12]:
# Untypeable K loci and STs Kleborate could only place as a nearest neighbour: the caveats
# the table's diversity counts carry.
caveats = genomes[(genomes["K_locus_confidence"] != "Typeable") | genomes["ST"].str.contains("LV")]
caveats[["isolate", "species", "ST", "K_locus", "K_locus_confidence", "O_locus", "contigs"]]

,isolate,species,ST,K_locus,K_locus_confidence,O_locus,contigs
library,,,,,,,
INF034_v2,INF034,Klebsiella pneumoniae,ST15-1LV,KL24,Typeable,OL2α.1,2
INF181,INF181,Klebsiella variicola subsp. variicola,ST347,KL14,Untypeable,OL5,3
INF311,INF311,Klebsiella pneumoniae,ST240,KL144,Untypeable,OL2α.1,4
KSB1_3C,KSB1_3C,Klebsiella variicola subsp. variicola,ST3198,KL103,Untypeable,OL10,2


In [13]:
# Outputs. per_isolate.tsv is the evidence for every number above; the other three are the
# tables the manuscript quotes.
genomes.reset_index().to_csv("per_isolate.tsv", sep="\t", index=False)
spread.rename_axis("metric").to_csv("sequencing_and_assembly_stats.tsv", sep="\t")
species.rename_axis("species").to_csv("species_table.tsv", sep="\t")
diversity.sort_values(["type", "n_isolates"], ascending=[True, False]).to_csv(
    "diversity_counts.tsv", sep="\t", index=False)

sorted(path.name for path in Path().glob("*.tsv"))

['diversity_counts.tsv',
 'per_isolate.tsv',
 'sequencing_and_assembly_stats.tsv',
 'species_table.tsv']